# Contrastive fine-tune CLIP on (image, caption) pairs

Uses CLIP's native training objective -- match each image to its correct caption in a
batch, push apart the rest. `CLIPModel` computes this loss internally via `return_loss=True`,
no hand-written contrastive math needed.

**Before running**: if your dataset is under ~200-300 pairs, a linear probe
(`Temple_Linear_Probe.ipynb`) will likely work better -- contrastive learning needs batch
diversity (enough negatives per step) that a small dataset can't really provide.

**Saves to a separate checkpoint, not the production model** -- this fine-tune only ever
sees devotional content, so it shouldn't replace the shared CLIP instance that also handles
safety moderation (catastrophic forgetting risk, discussed earlier). Deploy this as an
additional model used only for the devotional-scoring path.

In [ ]:
!pip install -q "transformers==4.46.3"

## 1. Load your (image, caption) manifest
Expects a JSONL file, one row per pair: `{"file": "image.jpg", "caption": "..."}` -- same
format the annotation notebooks (`Claude_Image_Annotation.ipynb` / `QwenVL_Image_Annotation.ipynb`)
produce (`short_caption` or `detailed_caption`), or your own.

**Before training**: same lesson as the Gemini dataset -- make sure these are real
photographs with verified captions, not synthetic images or unverified auto-generated
labels, before spending a training run on them.

In [ ]:
# EDIT ME
IMAGE_DIR = "/content/images"
MANIFEST_PATH = "/content/captions.jsonl"  # {"file": ..., "caption": ...} per line

import json, os

pairs = []
with open(MANIFEST_PATH) as f:
    for line in f:
        row = json.loads(line)
        path = os.path.join(IMAGE_DIR, row["file"])
        caption = row.get("caption") or row.get("short_caption")
        if os.path.exists(path) and caption:
            pairs.append((path, caption))

print(f"{len(pairs)} valid (image, caption) pairs")
if len(pairs) < 300:
    print("WARNING: under ~300 pairs -- consider a linear probe instead of contrastive fine-tuning.")

## 2. Load base CLIP + set up training

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "openai/clip-vit-base-patch16"  # match production, so the fine-tune is a drop-in swap

processor = CLIPProcessor.from_pretrained(BASE_MODEL, use_fast=False)
model = CLIPModel.from_pretrained(BASE_MODEL).to(DEVICE)

class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        path, caption = self.pairs[idx]
        return Image.open(path).convert("RGB"), caption

def collate(batch):
    images, captions = zip(*batch)
    enc = processor(
        text=list(captions), images=list(images),
        return_tensors="pt", padding=True, truncation=True,
    )
    return {k: v.to(DEVICE) for k, v in enc.items()}

BATCH_SIZE = 32  # keep this as large as your GPU allows -- contrastive loss quality depends
                 # on negatives per batch, small batches teach much less per step
train_loader = DataLoader(PairDataset(pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

In [ ]:
EPOCHS = 5
LR = 1e-6  # deliberately small -- a large LR on a pretrained model destroys existing
           # representations fast; contrastive fine-tuning should nudge, not overwrite

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(**batch, return_loss=True)  # CLIPModel computes the contrastive loss internally
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  mean_loss={total_loss/len(train_loader):.4f}")

model.eval()

## 3. Save as a separate checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/clip_devotional_finetuned"
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

## 4. Measure -- did it actually help?
Reuse the same eval harness from `CLIP_Model_Comparison_and_Fusion.ipynb` (Parts A/B): run
this fine-tuned checkpoint through the same `CLUSTER_PROMPTS` zero-shot scoring against your
labeled eval set, compare top-1 accuracy against the stock `openai/clip-vit-base-patch16`
baseline. Don't deploy this on the strength of a lower training loss alone -- lower loss on
the training pairs doesn't guarantee better zero-shot classification against your actual
cluster taxonomy, that has to be measured directly, same as every other change in this plan.